# 从备份压缩包中提取指定数据

该笔记本用于从 `data/backup/` 目录中的 `.tar.gz` 压缩包中按交易所、日期、币种提取并合并对应的数据。

In [ ]:
import pandas as pd
import tarfile
import io
from pathlib import Path

def load_orderbook_from_tar(tar_path, target_exchanges, target_types, symbol_map):
    """
    调整后的加载函数：支持外部参数控制过滤逻辑
    :param tar_path: tar.gz 归档文件路径
    :param target_exchanges: 列表，如 ['binance', 'okx']
    :param target_types: 列表，如 ['spot', 'swap']
    :param symbol_map: 字典，定义每个交易所对应的币种，如 {'binance': ['BTC_USDT'], ...}
    """
    print(f"正在加载压缩包数据：{tar_path} ...")
    dfs = []
    
    with tarfile.open(tar_path, 'r:gz') as tf:
        for m in tf.getmembers():
            if not m.isfile() or not m.name.endswith('.parquet'):
                continue
            
            # 路径解析 /orderbooks/market_type={type}/exchange={exchange}/symbol={symbol}/date={date}/{hour}.parquet
            # 或者         /trades/market_type={type}/exchange={exchange}/... 
            name_parts = m.name.lstrip('/').split('/')
            if len(name_parts) < 6:
                continue
            
            try:
                # 寻找表示业务线的第一层目录
                if 'orderbooks' in name_parts:
                    idx = name_parts.index('orderbooks')
                elif 'trades' in name_parts:
                    idx = name_parts.index('trades')
                else:
                    continue
                
                # 解析包含键值对的目录名
                m_type = name_parts[idx + 1].split('=')[1]     # market_type=spot
                m_exchange = name_parts[idx + 2].split('=')[1] # exchange=binance
                m_symbol = name_parts[idx + 3].split('=')[1]   # symbol=BTC_USDT
                m_date = name_parts[idx + 4].split('=')[1]     # date=2026-02-24
                
                # --- 动态过滤逻辑 ---
                # 1. 检查市场类型 (spot/swap)
                if m_type not in target_types:
                    continue
                
                # 2. 检查交易所
                if m_exchange not in target_exchanges:
                    continue
                
                # 3. 检查该交易所对应的币种是否匹配
                valid_symbols = symbol_map.get(m_exchange, [])
                if m_symbol not in valid_symbols:
                    continue
                    
            except (IndexError, ValueError) as e:
                continue

            # 读取 parquet 数据 (适配项目现状)
            fobj = tf.extractfile(m)
            if fobj is not None:
                bio = io.BytesIO(fobj.read())
                try:
                    df = pd.read_parquet(bio)
                    
                    df['exchange'] = m_exchange
                    df['symbol'] = m_symbol
                    df['market_type'] = m_type
                    df['date'] = m_date
                    dfs.append(df)
                except Exception as e:
                    # 某些文件可能无法读取，忽略或记录错误
                    # print(f"[ERROR] 读取文件 {m.name} 失败: {e}")
                    pass
                
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


def get_all_data(start_date, end_date, data_dir, exchanges, types, symbol_map):
    all_dfs = []
    date_range = pd.date_range(start=start_date, end=end_date)
    data_dir_path = Path(data_dir)
    
    for dt in date_range:
        date_str = dt.strftime('%Y-%m-%d')
        # 匹配对应日期带有时间戳的归档文件 crypto_data_YYYY-MM-DD_HHMMSS.tar.gz
        matching_files = list(data_dir_path.glob(f'crypto_data_{date_str}_*.tar.gz'))
        
        if matching_files:
            for tar_path in matching_files:
                # 将配置参数传入
                df = load_orderbook_from_tar(tar_path, exchanges, types, symbol_map)
                if not df.empty:
                    all_dfs.append(df)
        else:
            print(f"[WARN] 未找到日期 {date_str} 的数据文件")
    
    if not all_dfs:
        return pd.DataFrame()

    final_df = pd.concat(all_dfs, ignore_index=True)
    print(f"数据加载完成，总共读取了 {len(final_df)} 行数据。")
    return final_df


def clean_data(df):
    """
    基础数据清洗：
    1. 缺失值处理：前向填充 (ffill) 或后向填充
    2. 删除冗余列：如 local_ts, exchange_ts，整合并重新排序
    """
    print("开始执行数据清洗...")
    if df.empty:
        return df
    
    # 统一列名的小写处理（参照pipeline做法）
    df.columns = [str(c).strip().lower() for c in df.columns]
    
    # 1. 填补缺失值 (防止警告，使用 ffill / bfill 方法处理)
    try:
        df.ffill(inplace=True)
        df.bfill(inplace=True)
    except AttributeError:
        df = df.ffill().bfill()
    
    # 时序连续性排序处理
    if 'timestamp' in df.columns:
        df.sort_values(['exchange', 'timestamp'], inplace=True)
        df.reset_index(drop=True, inplace=True)
    elif 'exchange_ts' in df.columns:
        df['timestamp'] = df['exchange_ts']
        df.sort_values(['exchange', 'timestamp'], inplace=True)
        df.reset_index(drop=True, inplace=True)
    
    # 2. 删除不必要的列（如 local_ts, exchange_ts 等冗余时间戳）
    drop_cols = [c for c in ['local_ts', 'exchange_ts'] if c in df.columns]
    df.drop(columns=drop_cols, inplace=True, errors='ignore')

    front_cols = [c for c in ['timestamp', 'datetime', 'exchange', 'symbol', 'market_type', 'date'] if c in df.columns]
    other_cols = [c for c in df.columns if c not in front_cols]

    df = df[front_cols + other_cols]

    return df

In [ ]:
# 参数配置区域
START_DATE = '2026-02-24'
END_DATE = '2026-02-28'
DATA_DIR = '../data/backup' # 由于此文件在 src 目录，相对路径至上一级的 data/backup
TARGET_EXCHANGES = ['binance', 'okx']
TARGET_TYPES = ['spot']
SYMBOL_CONFIG = {
    'binance': ['BTC_USDT'],
    'okx': ['BTC_USDT']
}

raw_df = get_all_data(
    start_date=START_DATE, 
    end_date=END_DATE, 
    data_dir=DATA_DIR,
    exchanges=TARGET_EXCHANGES,
    types=TARGET_TYPES,
    symbol_map=SYMBOL_CONFIG
)

if not raw_df.empty:
    df = clean_data(raw_df) 
    print("\n清洗后数据预览：")
    display(df.head())
else:
    print("\n提取的数据为空。请检查参数是否正确。")